# FoodExpress · Fine-tune Qwen2.5-3B-Instruct (QLoRA)

Fine-tunes a ~3B model to be the FoodExpress menu assistant, then pushes it to **your** Hugging Face and exports it for **Ollama**.

**Before you run:** `Runtime → Change runtime type → T4 GPU` (free). Total time ~20-40 min.


## 1. Install dependencies


In [ ]:
!pip -q install transformers==4.46.3 trl==0.12.2 peft==0.14.0 bitsandbytes==0.45.0 accelerate==1.2.1 datasets==3.2.0 huggingface_hub>=0.26.0


## 2. Get the training scripts + build the dataset
Pulls the menu from your live FoodExpress API so the order-JSON uses real IDs (falls back to a bundled snapshot if the API is asleep).


In [ ]:
!wget -q https://raw.githubusercontent.com/gaurmaitreyi-png/foodexpress/main/finetune/build_dataset.py -O build_dataset.py
!wget -q https://raw.githubusercontent.com/gaurmaitreyi-png/foodexpress/main/finetune/finetune_qlora.py -O finetune_qlora.py
!python build_dataset.py
print(open('train.jsonl').readline())


## 3. Sign in to Hugging Face
Paste a token with **write** access from https://huggingface.co/settings/tokens . It stays in your Colab session — nobody else sees it.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


## 4. Fine-tune + push to your Hugging Face
Change `HF_REPO` to your username. The repo is created automatically.


In [ ]:
HF_REPO = 'YOUR_HF_USERNAME/foodexpress-qwen2.5-3b'  # <-- edit this
!python finetune_qlora.py --epochs 3 --merge --push --hf_repo $HF_REPO


## 5. Quick sanity test
Load the merged model and check it produces order JSON.


In [ ]:
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer
m_dir='foodexpress-qwen2.5-3b-lora-merged'
tok=AutoTokenizer.from_pretrained(m_dir)
model=AutoModelForCausalLM.from_pretrained(m_dir, torch_dtype=torch.bfloat16, device_map='auto')
msgs=[{'role':'system','content':'You are the FoodExpress menu assistant.'},
      {'role':'user','content':'Recommend something spicy and vegetarian.'}]
ids=tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
out=model.generate(ids, max_new_tokens=120, do_sample=False)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))


## 6. Export to GGUF for Ollama (optional, run on CPU/Colab)
This is how you actually plug it into FoodExpress (which already supports Ollama).
See `finetune/export_and_use_in_foodexpress.md` in the repo for the full steps.


In [ ]:
# Convert the merged model to GGUF using llama.cpp, then download it.
!git clone -q https://github.com/ggerganov/llama.cpp
!pip -q install -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py foodexpress-qwen2.5-3b-lora-merged --outfile foodexpress-qwen2.5-3b.gguf --outtype q8_0
from google.colab import files; files.download('foodexpress-qwen2.5-3b.gguf')
